In [1]:
import sys, os, csv, time, random
from pathlib import Path

_cwd = Path.cwd().resolve()
_review_hotel = next(
    (p for p in [_cwd, *_cwd.parents] if (p / "crawl_data_review.py").is_file()),
    None,
)
if _review_hotel is None:
    raise RuntimeError("Open/run notebook from src/scrapy_data/review_hotel or repo tree.")
sys.path.insert(0, str(_review_hotel))

from crawl_data_review import AgodaReviewCrawler, get_crawled_hotel_ids, merge_all_reviews
from config import HOTEL_URLS_FILE, REVIEWS_OUTPUT_FILE, get_random_delay

import pandas as pd

In [2]:
with open(HOTEL_URLS_FILE, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    hotels = list(reader)

print(f'Loaded {len(hotels)} hotels')

# Check checkpoint
crawled_ids = get_crawled_hotel_ids()
remaining = [h for h in hotels if h.get('id', '') not in crawled_ids]

print(f'Already crawled: {len(crawled_ids)}')
print(f'Remaining: {len(remaining)}')

Loaded 402 hotels
Already crawled: 402
Remaining: 0


In [3]:
crawler = AgodaReviewCrawler()
total_reviews = 0

try:
    for i, hotel in enumerate(remaining, 1):
        hotel_id = hotel.get('id', '')
        hotel_name = hotel.get('name', 'Unknown')
        
        print(f'\n[{i}/{len(remaining)}] {hotel_name} (ID: {hotel_id})')
        
        reviews = crawler.crawl_hotel_reviews(hotel)
        
        if reviews:
            crawler.save_hotel_reviews(reviews, hotel_id)
            total_reviews += len(reviews)
        else:
            # Save empty file to mark as crawled
            crawler.save_hotel_reviews([], hotel_id)
        
        # Delay between hotels
        if i < len(remaining):
            # Every 20 hotels: long rest (2-5 min)
            if i % 20 == 0:
                rest = random.uniform(120, 300)
                print(f'  🛑 Long rest: waiting {rest:.0f}s ({rest/60:.1f} min) after {i} hotels...')
                time.sleep(rest)
            # Every 5 hotels: batch rest (30-60s)
            elif i % 5 == 0:
                rest = random.uniform(30, 60)
                print(f'  🔄 Batch rest: waiting {rest:.0f}s after {i} hotels...')
                time.sleep(rest)
            # Normal delay between hotels
            else:
                delay = get_random_delay() * 3
                print(f'  Waiting {delay:.1f}s...')
                time.sleep(delay)

except KeyboardInterrupt:
    print(f'\nStopped! Collected {total_reviews} reviews. Re-run to resume.')

except Exception as e:
    print(f'\nError: {e}')
    import traceback
    traceback.print_exc()

finally:
    crawler.close()
    crawled = get_crawled_hotel_ids()
    print(f'\nTotal reviews this run: {total_reviews}')
    print(f'Hotels crawled: {len(crawled)}/{len(hotels)}')

Initializing Chrome WebDriver...



Total reviews this run: 0
Hotels crawled: 402/402


In [4]:
all_reviews = merge_all_reviews()
print(f'Total reviews: {len(all_reviews) if all_reviews else 0}')

✓ Merged 41344 reviews from 402 hotels into reviews_output.csv


Total reviews: 41344


In [5]:
df = pd.read_csv(REVIEWS_OUTPUT_FILE)
print(f'Shape: {df.shape}')
print(f'Hotels: {df["hotel_id"].nunique()}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nSample:')
df.head(10)

Shape: (41344, 4)
Hotels: 392

Columns: ['review_id', 'hotel_id', 'review_text', 'rating']

Sample:


,review_id,hotel_id,review_text,rating
0,1065527135,1001042,Sansan thật tuyệt vời và đó là nơi tôi luôn ch...,10.0
1,1042633845,1001042,Địa điểm tại đây khá thuận tiện cho khách du l...,8.0
2,970993281,1001042,Tôi rất thích khách sạn này. Nó nằm ở một con ...,10.0
3,889877242,1001042,trời ơi đi du lịch 1 mình từ lâu nhưng bà lễ t...,10.0
4,886199161,1001042,Tôi đã nhận được một số ưu đãi từ Agoda và có ...,10.0
5,911537767,1001042,Tôi đã ở nhiều khách sạn trong khu vực và khác...,9.0
6,1044862649,1001042,"Khách sạn sạch sẽ, thoáng mát, nhân viên nhiệt...",9.0
7,838535049,1001042,"Đặt kỳ vọng trước khi đặt phòng, đây là một kh...",9.0
8,931256741,1001042,Tài sản thì tuyệt vời trên tất cả các phương d...,10.0
9,1071090374,1001042,"Gần bãi biển, xung quanh chỗ ở có nhiều nhà hà...",8.0
